# A.X-Encoder 평가 v3 — 1000행 대응

`training_dataset_v3_1000_simple.csv` 기준. **100행 파일도 그대로 넣을 수 있다** —
행 수를 보고 하이퍼파라미터가 자동으로 바뀐다.

## `_simple.csv`를 써도 되는 이유

full(`training_dataset_v3_1000.csv`)과 **history·response 1000행이 전부 동일**하다.
빠진 건 `source` / `source_type` / `difficulty` 세 컬럼뿐이고, 이건 학습에 안 쓴다
(프로젝트 학습 규칙 2번: "입력은 history+response만").
`pair_id` 그룹 구조도 두 파일이 완전히 같다.

> 단, **`difficulty`별 성능 분리 보고는 simple로는 못 한다.** hard 56행이 무너지는지
> 보려면 나중에 full로 한 번 더 돌려야 한다.

## 100행과 무엇이 다른가

| | 100행 | v3 1000행 |
|---|---|---|
| response-only 기준선 | 0.940 (누출) | **0.400** (통제) |
| history-only | 0.500 | 0.264 |
| 같은 response가 양쪽 라벨에 | 0 / 100 | **500 / 500** |
| epoch당 스텝 | 10 | **100** |

**핵심은 마지막 두 줄이다.** v3는 고유 response 500개 전원이 적절·부적절 양쪽에
등장하므로 응답만 보고는 원리적으로 못 맞힌다. 여기서 성능이 나오면 그건
**문맥을 읽었다는 뜻**이고, 그게 이 프로젝트가 증명하려던 것이다.

## 자동 조정

행 수가 10배가 되면 필요한 epoch은 반대로 줄어든다(스텝 수가 이미 10배라서).
그대로 15 epoch을 돌리면 과적합에 시간만 40분 넘게 든다.

| 데이터 | EPOCHS | SEEDS | 예상 시간(T4) |
|---|---|---|---|
| < 300행 | 15 | 3개 | 약 8분 |
| ≥ 300행 | **4** | **1개** | **약 10분** |

1000행에서 seed를 1개로 줄이는 근거: fold당 검증이 **200행**이라 100행일 때(20행)보다
훨씬 안정적이다. 시간이 남으면 `SEEDS`에 7, 2026을 추가한다.


## 1. 환경 확인 — **설치하지 않는다**

Colab에는 transformers·torch·sklearn·pandas가 이미 들어 있다.
`pip install -U` 로 올리면 pandas가 2.x → 3.x로 메이저 업그레이드되면서
torch 임포트가 `circular import` 로 깨진다. 실제로 그렇게 깨졌다.

> 환경이 이미 손상됐다면 **런타임 → 런타임 연결 해제 및 삭제** 로 새 런타임을 받는다.
> "세션 다시 시작"만으로는 부족하다 — 잘못 설치된 패키지가 디스크에 남는다.


In [ ]:
# 설치하지 않는다. Colab 기본 환경에 전부 들어 있다.
# `pip install -U` 는 pandas/numpy를 메이저 업그레이드해 torch 임포트를 깨뜨린다.
def _ver(name):
    try:
        return getattr(__import__(name), "__version__", "?")
    except Exception as e:
        return f"!! {type(e).__name__}: {e}"

vers = {m: _ver(m) for m in ["numpy", "pandas", "sklearn", "torch", "transformers"]}
for m, v in vers.items():
    print(f"{m:14s} {v}")

broken = [m for m, v in vers.items() if str(v).startswith("!!")]
print()
if broken:
    print(f"[환경 손상] {', '.join(broken)} 임포트 실패.")
    print("  → 런타임 → '런타임 연결 해제 및 삭제' 후 새 런타임에서 이 셀부터 다시 실행.")
    print("  → pip install 은 하지 말 것.")
else:
    from packaging.version import Version
    ok = Version(vers["transformers"].split("+")[0]) >= Version("4.48")
    print(f"transformers {vers['transformers']} — ModernBERT 지원 {'OK' if ok else '부족'}")
    if not ok:
        print("  → !pip install -U transformers  (다른 패키지는 건드리지 말 것)")
        print("  → 설치 후 '런타임 → 세션 다시 시작' 하고 이 셀부터 다시")
    import torch
    print(f"CUDA {torch.cuda.is_available()} | "
          f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 — 런타임 유형을 T4로'}")

numpy          2.0.2
pandas         2.2.2
sklearn        1.6.1
torch          2.11.0+cu128
transformers   5.13.1

transformers 5.13.1 — ModernBERT 지원 OK
CUDA True | Tesla T4


In [ ]:
CSV_PATH  = "training_dataset_v3_1000_simple.csv"
GROUP_BY  = "pair"   # "pair" = 프로젝트 규칙(pair_id 단위) | "strict" = pair+history 연결 성분

import os, numpy as np, pandas as pd
if not os.path.exists(CSV_PATH):
    from google.colab import files
    CSV_PATH = list(files.upload().keys())[0]

df = pd.read_csv(CSV_PATH)
assert not ({"pair_id","history","response","label"} - set(df.columns)), f"컬럼 누락: {list(df.columns)}"
df = df.dropna(subset=["history","response","label"]).reset_index(drop=True)
df["history"]  = df["history"].astype(str)
df["response"] = df["response"].astype(str)

# --- 화자 익명화 (운영 경로와 형식을 맞춘다) --------------------------------
# 학습 데이터의 history는 "선임:" "나:" "팀장:" 같은 실제 호칭을 쓴다(66종).
# 그런데 웹앱은 FR-6.2에 따라 화자를 A/B/C로 익명화해서 모델에 넣는다.
# 그대로 학습하면 배포 시 **학습에서 한 번도 못 본 형식**이 들어간다.
# anonymize.label_speakers 와 같은 규칙(등장 순서 → A,B,C…)으로 미리 바꾼다.
import re
ANONYMIZE_SPEAKERS = True

def _anonymize(h: str) -> str:
    AL = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    seen, out = {}, []
    for line in str(h).replace("\r\n", "\n").split("\n"):
        mo = re.match(r"^([^:]{1,10}):\s*(.*)$", line)
        if not mo:
            out.append(line); continue
        who, txt = mo.group(1), mo.group(2)
        if who not in seen:
            seen[who] = AL[len(seen)] if len(seen) < 26 else f"S{len(seen)}"
        out.append(f"{seen[who]}: {txt}")
    return "\n".join(out)

if ANONYMIZE_SPEAKERS:
    before = df["history"].iloc[0][:56]
    df["history"] = df["history"].map(_anonymize)
    print("\n[익명화] 화자 호칭 → A/B/C (운영 프롬프트와 동일 규칙)")
    print(f"  전: {before}...")
    print(f"  후: {df['history'].iloc[0][:56]}...")

y = (df["label"].astype(str).str.strip() == "부적절").astype(int).values

if GROUP_BY == "strict":
    # pair_id와 history를 같은 그룹으로 묶는다(union-find). 같은 방의 여러 응답이
    # 학습·검증에 갈라지는 것까지 막는 더 엄격한 분할.
    par = {}
    def find(x):
        par.setdefault(x, x)
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: par[ra] = rb
    for p, h in zip(df["pair_id"].astype(str), df["history"]):
        union("P:" + p, "H:" + h)
    groups = np.array([find("P:" + p) for p in df["pair_id"].astype(str)])
else:
    groups = df["pair_id"].astype(str).values

print(f"{CSV_PATH} | {len(df)}행 | 부적절 {y.sum()} / 적절 {(1-y).sum()}")
print(f"그룹 방식 '{GROUP_BY}' → 그룹 {len(set(groups))}개 (평균 {len(df)/len(set(groups)):.1f}행)")

Saving training_dataset_v3_1000_simple.csv to training_dataset_v3_1000_simple.csv

[익명화] 화자 호칭 → A/B/C (운영 프롬프트와 동일 규칙)
  전: 선임: 진행하면서 새로 확인한 내용이 있어요.
나: 문제는 해결됐나요?
선임: 어제 배포한 버전은...
  후: A: 진행하면서 새로 확인한 내용이 있어요.
B: 문제는 해결됐나요?
A: 어제 배포한 버전은 현재까...
training_dataset_v3_1000_simple.csv | 1000행 | 부적절 500 / 적절 500
그룹 방식 'pair' → 그룹 500개 (평균 2.0행)


## 2. 누출 기준선 — 이번엔 **낮게** 나와야 정상이다

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score

def surface(texts, name):
    pipe = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), min_df=1),
                         LogisticRegression(max_iter=2000))
    pred = cross_val_predict(pipe, np.array(texts), y,
                             cv=StratifiedGroupKFold(5, shuffle=True, random_state=42), groups=groups)
    a = accuracy_score(y, pred); print(f"{name:18s} {a:.3f}"); return a

BASE_RESP = surface(df["response"], "response-only")
BASE_HIST = surface(df["history"],  "history-only")
both = (df.groupby("response")["label"].nunique() > 1).sum()
print(f"\n양쪽 라벨에 등장하는 response: {both} / {df['response'].nunique()}")

LEAKY = BASE_RESP > 0.65
if LEAKY:
    # 누출 데이터: 정답이 표면에 있으므로 학습이 정상이면 TF-IDF 근처까지 가야 한다
    PASS_BAR = max(0.70, BASE_RESP - 0.05)
    print(f"\n[누출 데이터] 학습 합격선 {PASS_BAR:.3f} — 표면 패턴조차 못 맞히면 파이프라인 고장")
else:
    # 통제 데이터: 표면으로는 못 맞히므로 합격선은 '우연 이상'
    PASS_BAR = 0.60
    print(f"\n[통제 데이터] 응답 표면으로는 못 맞힌다. 합격선 {PASS_BAR:.3f}")
    print("  → 여기서 성능이 나오면 그건 문맥을 읽었다는 뜻이다.")

response-only      0.391
history-only       0.294

양쪽 라벨에 등장하는 response: 500 / 500

[통제 데이터] 응답 표면으로는 못 맞힌다. 합격선 0.600
  → 여기서 성능이 나오면 그건 문맥을 읽었다는 뜻이다.


## 3. 모델 · 하이퍼파라미터 (행 수에 따라 자동 조정)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "skt/A.X-Encoder-base"
LR = 5e-5
BATCH = 8

# 행이 많아지면 epoch당 스텝이 이미 충분하므로 EPOCHS를 줄인다.
if len(df) < 300:
    EPOCHS, SEEDS, MAX_LEN = 15, [42, 7, 2026], 256
else:
    EPOCHS, SEEDS, MAX_LEN = 4, [42], 384

steps_per_epoch = int(np.ceil(len(df) * 0.8 / BATCH))
print(f"{len(df)}행 → EPOCHS={EPOCHS} | SEEDS={SEEDS} | MAX_LEN={MAX_LEN} | LR={LR}")
print(f"fold당 {steps_per_epoch}스텝/epoch × {EPOCHS} = 총 {steps_per_epoch*EPOCHS}스텝")
print(f"학습 횟수 {5*len(SEEDS)}회")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    try:
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2, attn_implementation="sdpa").to(device)
    except Exception:
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2).to(device)

def encode(h, r):
    return tokenizer(list(h), list(r), truncation="only_first",
                     max_length=MAX_LEN, padding=True, return_tensors="pt")

lens = [len(tokenizer(h, r)["input_ids"]) for h, r in zip(df["history"][:400], df["response"][:400])]
print(f"\n토큰 길이(표본 400) 중앙 {int(np.median(lens))} | p95 {int(np.percentile(lens,95))} | 최대 {max(lens)}")
print(f"MAX_LEN={MAX_LEN} → {'전부 들어간다' if max(lens) <= MAX_LEN else '일부 잘림 (history 앞부분부터)'}")

1000행 → EPOCHS=4 | SEEDS=[42] | MAX_LEN=384 | LR=5e-05
fold당 100스텝/epoch × 4 = 총 400스텝
학습 횟수 5회


config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/969 [00:00<?, ?B/s]


토큰 길이(표본 400) 중앙 67 | p95 102 | 최대 117
MAX_LEN=384 → 전부 들어간다


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from transformers import get_linear_schedule_with_warmup

def train_one_fold(tr_idx, va_idx, seed, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = load_model()

    enc = encode(df["history"].values[tr_idx], df["response"].values[tr_idx])
    dl = DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"],
                                  torch.tensor(y[tr_idx], dtype=torch.long)),
                    batch_size=BATCH, shuffle=True)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, max(1, int(total*0.1)), total)
    amp = torch.bfloat16 if USE_BF16 else torch.float32

    losses = []
    model.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for ids, mask, lab in dl:
            ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            tot += loss.item()
        losses.append(tot/len(dl))
        if verbose:
            print(f"    epoch {ep+1}/{EPOCHS}  loss {losses[-1]:.4f}")

    @torch.no_grad()
    def predict(h, r):
        model.eval(); out = []
        for i in range(0, len(r), 64):
            e = encode(h[i:i+64], r[i:i+64])
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                lg = model(input_ids=e["input_ids"].to(device),
                           attention_mask=e["attention_mask"].to(device)).logits
            out.append(torch.softmax(lg.float(), -1)[:,1].cpu().numpy())
        return np.concatenate(out)

    # 학습셋 정확도는 표본 200행으로만 (1000행 전량은 시간 낭비)
    smp = tr_idx[:200]
    train_acc = (((predict(df["history"].values[smp], df["response"].values[smp]) > 0.5).astype(int))
                 == y[smp]).mean()

    h_va, r_va = df["history"].values[va_idx], df["response"].values[va_idx]
    res = {"normal":  predict(h_va, r_va),
           "swapped": predict(np.roll(h_va, 1), r_va),
           "empty":   predict(np.array([""]*len(r_va)), r_va),
           "train_acc": train_acc, "losses": losses}
    del model; torch.cuda.empty_cache()
    return res

## 4. 학습 점검 — 단일 fold

전체 CV를 돌리기 전에 여기서 먼저 본다. 1000행이면 약 2분.

**학습셋 정확도가 0.90을 못 넘으면 여기서 멈추고** `EPOCHS`를 올린다.


In [ ]:
import time
cv0 = StratifiedGroupKFold(5, shuffle=True, random_state=SEEDS[0])
tr0, va0 = next(iter(cv0.split(df, y, groups)))
print(f"학습 {len(tr0)}행 / 검증 {len(va0)}행")

t0 = time.time()
chk = train_one_fold(tr0, va0, SEEDS[0], verbose=True)
el = time.time() - t0

p = chk["normal"]
va_acc = (((p > 0.5).astype(int)) == y[va0]).mean()
decisive = ((p < 0.2) | (p > 0.8)).mean()

print(f"\n({el:.0f}초 · 전체 CV 예상 {el*5*len(SEEDS)/60:.1f}분)")
print(f"loss        : {chk['losses'][0]:.4f} → {chk['losses'][-1]:.4f}")
print(f"학습셋 정확도 : {chk['train_acc']:.3f}  (목표 ≥ 0.90)")
print(f"검증 정확도   : {va_acc:.3f}  (합격선 {PASS_BAR:.3f})")
print(f"확률 분포     : 표준편차 {p.std():.3f} | 확신 비율 {decisive:.0%}")

print("\n" + "="*60)
# §6과 같은 상대 기준을 쓴다. 학습셋 0.90은 통제 데이터에서 과적합해야 닿는 값이라,
# 절대 기준을 걸면 건강한 학습을 실패로 판정한다. 학습셋이 **검증보다도 못할 때**가
# 진짜 학습 부족이다.
if chk["train_acc"] < max(0.70, va_acc - 0.02) or va_acc < PASS_BAR:
    print(f"[학습 실패] 학습 데이터조차 못 맞힌다. EPOCHS를 {EPOCHS*2}로 올려 §3부터 재실행.")
elif decisive < 0.4:
    print("[결정 미흡] 확률이 0.5 근처에 몰려 있다. EPOCHS를 늘릴 것.")
else:
    print("[통과] 아래 전체 CV로 진행.")
print("="*60)

학습 800행 / 검증 200행


model.safetensors: reconstructing file:   0%|          |  0.00B /  299MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch 1/4  loss 0.7400
    epoch 2/4  loss 0.5718
    epoch 3/4  loss 0.3659
    epoch 4/4  loss 0.2509

(276초 · 전체 CV 예상 23.0분)
loss        : 0.7400 → 0.2509
학습셋 정확도 : 0.885  (목표 ≥ 0.90)
검증 정확도   : 0.890  (합격선 0.600)
확률 분포     : 표준편차 0.366 | 확신 비율 68%

[통과] 아래 전체 CV로 진행.


## 5. 전체 교차검증

In [ ]:
CONDS = ["normal", "swapped", "empty"]
oof = {s: {c: np.zeros(len(df)) for c in CONDS} for s in SEEDS}
train_accs = []

t0 = time.time()
for seed in SEEDS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    for k, (tr, va) in enumerate(cv.split(df, y, groups)):
        r = train_one_fold(tr, va, seed)
        for c in CONDS:
            oof[seed][c][va] = r[c]
        train_accs.append(r["train_acc"])
        print(f"  seed {seed} fold {k+1}/5  train_acc {r['train_acc']:.3f}  [{time.time()-t0:.0f}s]", end="\r")
    acc = ((oof[seed]["normal"] > 0.5).astype(int) == y).mean()
    print(f"seed {seed}: OOF acc = {acc:.3f}" + " "*30)
print(f"\n총 {time.time()-t0:.0f}초 | 평균 학습셋 정확도 {np.mean(train_accs):.3f}")

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 42: OOF acc = 0.854                              

총 1380초 | 평균 학습셋 정확도 0.892


## 6. 판정

학습 성공 여부를 **먼저** 확인하고, 통과했을 때만 절제 실험을 해석한다.

통제된 데이터(v3)에서는 `swapped`·`empty`가 **0.5 근처로 무너져야 정상**이다.
응답만 남기면 정답을 알 수 없게 설계된 데이터니까.


In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for c in CONDS:
    accs = [(((oof[s][c] > 0.5).astype(int)) == y).mean() for s in SEEDS]
    aucs = [roc_auc_score(y, oof[s][c]) for s in SEEDS]
    rows.append({"조건": c, "정확도": f"{np.mean(accs):.3f} ± {np.std(accs):.3f}",
                 "AUC": f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}", "_a": np.mean(accs)})
abl = pd.DataFrame(rows)
print(abl[["조건","정확도","AUC"]].to_string(index=False))

a_norm, a_swap, a_empty = abl["_a"].values
drop_swap = a_norm - a_swap
mean_train = np.mean(train_accs)
print(f"\nOOF {a_norm:.3f} | 합격선 {PASS_BAR:.3f} | 학습셋 {mean_train:.3f}")
print(f"swapped 하락폭 {drop_swap:+.3f} | empty 하락폭 {a_norm-a_empty:+.3f}")
print("="*66)

# 학습셋 기준은 절대값이 아니라 **상대값**이어야 한다. 통제된 데이터에서
# train_acc 0.90은 과적합해야만 닿는 수치라, 절대 기준을 걸면 건강한 학습을
# 실패로 판정한다. 학습셋이 검증보다도 못할 때가 진짜 학습 부족이다.
UNDERFIT = mean_train < max(0.70, a_norm - 0.02)
if a_norm < PASS_BAR or UNDERFIT:
    print("[학습 실패] 절제 실험을 해석하지 않는다.")
    print(f"  → EPOCHS {EPOCHS} → {EPOCHS*2}, LR {LR} → 8e-5 로 §3부터 재실행.")
elif a_empty > a_norm + 0.02:
    print("[이상] 문맥을 지웠을 때 성능이 더 높다. 학습이 불안정하다.")
elif drop_swap < 0.10:
    print("[문맥 미사용] history를 바꿔도 성능이 유지된다 — 응답 표면만 보고 있다.")
    if LEAKY:
        print("  → 누출 데이터에서는 예상된 결과다. v3 1000행으로 재실행할 것.")
    else:
        print("  → 통제 데이터인데 이 결과라면 학습이 부족한 쪽을 먼저 의심할 것.")
elif drop_swap < 0.30:
    print("[부분 사용] 문맥을 보긴 하지만 응답 표면 의존이 남아 있다.")
else:
    print("[문맥 사용] 문맥을 바꾸면 판정이 뒤집힌다. 의도한 대로 동작한다.")
    print(f"  어휘 baseline 0.725 대비 {a_norm-0.725:+.3f}")
print("="*66)

     조건           정확도           AUC
 normal 0.854 ± 0.000 0.903 ± 0.000
swapped 0.469 ± 0.000 0.481 ± 0.000
  empty 0.485 ± 0.000 0.477 ± 0.000

OOF 0.854 | 합격선 0.600 | 학습셋 0.892
swapped 하락폭 +0.385 | empty 하락폭 +0.369
[문맥 사용] 문맥을 바꾸면 판정이 뒤집힌다. 의도한 대로 동작한다.
  어휘 baseline 0.725 대비 +0.129


## 7. 성능 지표 · threshold

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)

p_mean = np.mean([oof[s]["normal"] for s in SEEDS], axis=0)
pred = (p_mean > 0.5).astype(int)
print(classification_report(y, pred, target_names=["적절","부적절"], digits=3))

cm = confusion_matrix(y, pred)
print("혼동행렬        예측:적절  예측:부적절")
print(f"  실제 적절      {cm[0,0]:5d}     {cm[0,1]:5d}   <- false positive")
print(f"  실제 부적절    {cm[1,0]:5d}     {cm[1,1]:5d}")
print(f"\n확률: 표준편차 {p_mean.std():.3f} | 확신 비율 {(((p_mean<0.2)|(p_mean>0.8)).mean()):.0%}")

print("\n=== 기준선 대조 ===")
print(pd.DataFrame([
    {"방법":"response-only (TF-IDF)","정확도":f"{BASE_RESP:.3f}"},
    {"방법":"history-only  (TF-IDF)","정확도":f"{BASE_HIST:.3f}"},
    {"방법":"어휘+격식 baseline(문서)","정확도":"0.725"},
    {"방법":"A.X-Encoder (empty)","정확도":f"{a_empty:.3f}"},
    {"방법":"A.X-Encoder (normal)","정확도":f"{a_norm:.3f}"},
]).to_string(index=False))

              precision    recall  f1-score   support

          적절      0.858     0.848     0.853       500
         부적절      0.850     0.860     0.855       500

    accuracy                          0.854      1000
   macro avg      0.854     0.854     0.854      1000
weighted avg      0.854     0.854     0.854      1000

혼동행렬        예측:적절  예측:부적절
  실제 적절        424        76   <- false positive
  실제 부적절       70       430

확률: 표준편차 0.383 | 확신 비율 73%

=== 기준선 대조 ===
                    방법   정확도
response-only (TF-IDF) 0.391
history-only  (TF-IDF) 0.294
    어휘+격식 baseline(문서) 0.725
   A.X-Encoder (empty) 0.485
  A.X-Encoder (normal) 0.854


In [ ]:
print("thr   precision  recall     F1     팝업률")
best = None
for thr in np.arange(0.05, 1.00, 0.05):
    pr = (p_mean > thr).astype(int)
    if pr.sum() == 0: continue
    pc, rc = precision_score(y, pr, zero_division=0), recall_score(y, pr, zero_division=0)
    f1 = f1_score(y, pr, zero_division=0)
    print(f"{thr:.2f}    {pc:.3f}     {rc:.3f}   {f1:.3f}    {pr.mean():.3f}")
    if best is None or f1 > best[3]: best = (thr, pc, rc, f1)
print(f"\nF1 최대: thr={best[0]:.2f} (precision {best[1]:.3f} / recall {best[2]:.3f})")

# 프로젝트 우선 지표는 precision. 0.95를 만족하는 최소 threshold를 찾는다.
hit = [(t, precision_score(y,(p_mean>t).astype(int),zero_division=0),
           recall_score(y,(p_mean>t).astype(int),zero_division=0))
       for t in np.arange(0.05,1.00,0.05)
       if (p_mean>t).sum()>0 and precision_score(y,(p_mean>t).astype(int),zero_division=0) >= 0.95]
print(f"precision ≥ 0.95 최소 thr = {hit[0][0]:.2f} (recall {hit[0][2]:.3f})" if hit
      else "precision 0.95 를 만족하는 threshold 없음")

out = df.copy()
out["prob_부적절"] = p_mean.round(3)
out["예측"] = np.where(pred==1, "부적절", "적절")
out["정답여부"] = np.where(pred==y, "O", "X")
out.to_csv("oof_predictions_v3.csv", index=False, encoding="utf-8-sig")
print(f"\n오분류 {(pred!=y).sum()}/{len(df)}행 | 저장: oof_predictions_v3.csv")

thr   precision  recall     F1     팝업률
0.05    0.607     0.984   0.751    0.810
0.10    0.662     0.960   0.784    0.725
0.15    0.692     0.938   0.796    0.678
0.20    0.721     0.920   0.808    0.638
0.25    0.745     0.906   0.818    0.608
0.30    0.768     0.896   0.827    0.583
0.35    0.798     0.886   0.840    0.555
0.40    0.817     0.882   0.848    0.540
0.45    0.831     0.866   0.848    0.521
0.50    0.850     0.860   0.855    0.506
0.55    0.857     0.850   0.853    0.496
0.60    0.867     0.836   0.851    0.482
0.65    0.871     0.810   0.839    0.465
0.70    0.882     0.776   0.826    0.440
0.75    0.880     0.722   0.793    0.410
0.80    0.900     0.668   0.767    0.371
0.85    0.921     0.582   0.713    0.316
0.90    0.916     0.434   0.589    0.237
0.95    0.933     0.250   0.394    0.134

F1 최대: thr=0.50 (precision 0.850 / recall 0.860)
precision 0.95 를 만족하는 threshold 없음

오분류 146/1000행 | 저장: oof_predictions_v3.csv


## 7-b. 기저율 보정 — **운영 threshold는 여기서 정한다**

데이터는 50:50이지만 실제 채팅에서 오발송은 드물다.
같은 모델이라도 기저율이 낮으면 precision이 급락한다.


In [ ]:
# --- 실사용 기저율 보정 -------------------------------------------------
# 이 데이터는 50:50이지만 실제 채팅에서 오발송은 훨씬 드물다.
# 같은 모델·같은 threshold라도 기저율이 낮으면 precision이 급락한다.
# 팝업이 뜨는 쪽의 대다수가 오탐이면 기능이 죽는다 — README의 우선 지표가 그 이유다.
print(f"{'thr':>5} {'FPR':>6} {'recall':>7} |{'  50:50':>9}{'   10%':>8}{'    5%':>8}{'    1%':>8}")
N_POS = int(y.sum())
for t in np.arange(0.50, 1.00, 0.05):
    pr = (p_mean > t).astype(int)
    if pr.sum() == 0:
        continue
    pc = precision_score(y, pr, zero_division=0)
    rc = recall_score(y, pr, zero_division=0)
    if pc == 0:
        continue
    fpr = (N_POS * rc) * (1 - pc) / pc / (len(y) - N_POS)
    row = f"{t:5.2f} {fpr:6.3f} {rc:7.3f} |{pc:9.3f}"
    for base in (0.10, 0.05, 0.01):
        denom = rc * base + fpr * (1 - base)
        row += f"{(rc*base/denom if denom else 0):8.3f}"
    print(row)

print("\n오발송이 실제로 몇 %인지가 threshold를 결정한다.")
print("50:50 기준 precision만 보고 운영 threshold를 정하면 팝업의 대부분이 오탐이 된다.")
print("→ 웹앱 judgment_logs 의 user_action(sent/cancelled)이 실측 기저율을 준다 (FR-7.5).")

  thr    FPR  recall |    50:50     10%      5%      1%


NameError: name 'y' is not defined

## 8. 발표용 대조표

100행과 1000행을 각각 돌린 뒤 아래를 채우면 슬라이드 한 장이 된다.

| | 100행 (팀 제작, 누출) | v3 1000행 (재조립) |
|---|---|---|
| response-only 기준선 | 0.940 | 0.400 |
| 같은 response 양쪽 라벨 | 0 / 100 | 500 / 500 |
| A.X-Encoder OOF | | |
| **swapped 하락폭** | | |
| 판정 | | |

**읽는 법** — 100행은 정확도가 높아도 `swapped` 하락폭이 0에 가깝고,
v3는 하락폭이 크게 나와야 한다. 그 대비가 곧
**"데이터를 왜 다시 만들어야 했는가"** 의 실증이다.

## 다음 단계

1. `difficulty`별 성능은 **full 파일**(`training_dataset_v3_1000.csv`)로 다시 돌려야 나온다.
   `hard` 56행이 무너지면 개체 추적 실패 — v2.5 방식으로 hard 데이터를 증분한다
2. 배포용 모델은 이 노트북이 아니라 `training/train_ax_encoder.py`로 만든다
   (전체 데이터 1회 학습 + `final_model/` + `threshold.json`)
3. threshold는 F1 최대가 아니라 **precision 0.95 기준**으로 잡는다 —
   멀쩡한 메시지에 팝업이 뜨는 쪽이 치명적이다


## 9. 배포용 최종 모델

§6이 `[문맥 사용]` 또는 `[부분 사용]`일 때만 실행한다.
전체 1000행으로 1회 학습해 저장한다 (약 6~8분).


In [ ]:
# =====================================================================
# §9. 배포용 최종 모델 — §6이 [문맥 사용] 또는 [부분 사용]일 때만 실행
# =====================================================================
# 지금까지의 학습은 전부 "평가용"이라 20%를 검증으로 빼고 돌렸고 매번 버렸다.
# 여기서는 전체 데이터로 한 번 학습해 저장한다.
import json, os, shutil
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import precision_score, recall_score

OUT = "context_checker"
TARGET_PRECISION = 0.95   # 프로젝트 우선 지표. false positive가 치명적이다.

# --- 1) threshold 는 OOF에서 뽑는다 ------------------------------------
# 최종 모델이 자기 학습 데이터에 매긴 점수로 정하면 낙관적으로 편향된다.
# §6의 p_mean은 fold 밖 예측이라 편향이 없다.
cands = []
for t in np.arange(0.05, 1.00, 0.01):
    pr = (p_mean > t).astype(int)
    if pr.sum() == 0:
        continue
    cands.append((t, precision_score(y, pr, zero_division=0),
                     recall_score(y, pr, zero_division=0), pr.mean()))

hit = [c for c in cands if c[1] >= TARGET_PRECISION]
if hit:
    thr, prec, rec, flag = hit[0]
else:
    thr, prec, rec, flag = max(cands, key=lambda c: c[1])
    print(f"[주의] precision {TARGET_PRECISION} 를 만족하는 지점이 없다.")

print(f"선택 threshold {thr:.2f} — precision {prec:.3f} / recall {rec:.3f} / 팝업률 {flag:.3f}")
if rec < 0.5:
    print(f"  [!] recall {rec:.3f} — 오발송 {1-rec:.0%}를 놓친다.")
    print("      precision을 위해 recall을 버린 지점이다. 운영 전에 재검토할 것.")

# 이 데이터는 50:50이다. 실사용 기저율이 낮으면 같은 threshold의 precision이
# 크게 떨어진다. 지금 하나로 못 박지 말고 **스윕 전량을 같이 저장**해,
# 나중에 실사용 로그로 기저율을 알게 됐을 때 재학습 없이 다시 고를 수 있게 한다.
N_POS = int(y.sum()); N_NEG = len(y) - N_POS
sweep_rows = []
for t, pc, rc, fl in cands:
    if pc <= 0:
        continue
    fpr = (N_POS * rc) * (1 - pc) / pc / N_NEG
    row = {"threshold": round(float(t), 3), "precision_balanced": round(float(pc), 4),
           "recall": round(float(rc), 4), "flag_rate": round(float(fl), 4),
           "fpr": round(float(fpr), 4)}
    for base in (0.10, 0.05, 0.01):
        den = rc * base + fpr * (1 - base)
        row[f"precision_at_{int(base*100)}pct"] = round(float(rc * base / den), 4) if den else 0.0
    sweep_rows.append(row)

# --- 2) 전체 데이터로 1회 학습 -----------------------------------------
torch.manual_seed(42); np.random.seed(42)
model = load_model()
enc = encode(df["history"].values, df["response"].values)
dl = DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"],
                              torch.tensor(y, dtype=torch.long)),
                batch_size=BATCH, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = len(dl) * EPOCHS
sch = get_linear_schedule_with_warmup(opt, max(1, int(total*0.1)), total)
amp = torch.bfloat16 if USE_BF16 else torch.float32

model.train()
for ep in range(EPOCHS):
    tot = 0.0
    for ids, mask, lab in dl:
        ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
        with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step(); opt.zero_grad()
        tot += loss.item()
    print(f"  epoch {ep+1}/{EPOCHS}  loss {tot/len(dl):.4f}")

# --- 3) 저장 -----------------------------------------------------------
if os.path.exists(OUT):
    shutil.rmtree(OUT)
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

with open(f"{OUT}/threshold.json", "w", encoding="utf-8") as f:
    json.dump({
        "threshold": round(float(thr), 4),
        "target_precision": TARGET_PRECISION,
        "oof_precision": round(float(prec), 4),
        "oof_recall": round(float(rec), 4),
        "oof_flag_rate": round(float(flag), 4),
        "max_len": MAX_LEN,          # 추론 때 반드시 같아야 한다
        "epochs": EPOCHS, "lr": LR, "batch": BATCH,
        "n_rows": int(len(df)), "csv": CSV_PATH,
        "base_model": MODEL_NAME,
        "label_positive": "부적절",   # index 1 이 부적절이다
        # 실사용 기저율을 알게 되면 재학습 없이 여기서 threshold를 다시 고른다.
        "sweep": sweep_rows,
    }, f, ensure_ascii=False, indent=2)

# --- 4) 저장한 것을 그대로 다시 불러 검산 -------------------------------
# save/load 경로가 실제로 도는지 여기서 확인한다. 로컬에 가져가서 깨지면
# 원인을 찾기 훨씬 어렵다.
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tk = AutoTokenizer.from_pretrained(OUT)
md = AutoModelForSequenceClassification.from_pretrained(OUT).to(device).eval()
HIST = ("A: 내일 배포 리허설 몇 시로 할까요?\nB: 오전 10시면 될 것 같습니다\n"
        "C: 저는 10시 좋습니다. 스테이징 먼저 올릴게요\nA: 롤백 절차도 한 번 봐주세요")
for cand, expect in [("오늘 저녁에 치킨 어때? 양념 vs 후라이드", "부적절"),
                     ("롤백은 alembic downgrade 한 단계로 정리해뒀습니다", "적절")]:
    e = tk(HIST, cand, truncation="only_first", max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        p = torch.softmax(md(**e).logits.float(), -1)[0, 1].item()
    got = "부적절" if p >= thr else "적절"
    print(f"  {'OK ' if got == expect else 'X  '} p={p:.3f} → {got} (기대 {expect}) | {cand[:26]}")

# --- 5) 내려받기 -------------------------------------------------------
!zip -qr {OUT}.zip {OUT}
print(f"\n{OUT}.zip  {os.path.getsize(OUT + '.zip')/1e6:.0f}MB")
# 300MB 다운로드는 브라우저에서 끊기기 쉽다. 실패하면 Drive 경유로 바꾼다.
try:
    from google.colab import files
    files.download(f"{OUT}.zip")
except Exception as exc:
    print(f"직접 다운로드 실패({type(exc).__name__}). Drive로 옮긴다:")
    print("  from google.colab import drive; drive.mount('/content/drive')")
    print(f"  !cp {OUT}.zip /content/drive/MyDrive/")


선택 threshold 0.99 — precision 1.000 / recall 0.034 / 팝업률 0.017
  [!] recall 0.034 — 오발송 97%를 놓친다.
      precision을 위해 recall을 버린 지점이다. 운영 전에 재검토할 것.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/4  loss 0.7477
  epoch 2/4  loss 0.4985
  epoch 3/4  loss 0.2639
  epoch 4/4  loss 0.1620


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

  X   p=0.989 → 적절 (기대 부적절) | 오늘 저녁에 치킨 어때? 양념 vs 후라이드
  OK  p=0.020 → 적절 (기대 적절) | 롤백은 alembic downgrade 한 단계

context_checker.zip  227MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>